# Гибридная система рекомендаций книг (goodbooks-10k)

Лабораторный журнал решения: вызовы функций пакета `recsys` (`src/recsys`), промежуточные таблицы и графики по этапам. Постановка, обоснования и итоговые выводы — в `README.md`. Графики, используемые в README, сохраняются в `reports/figures/`.


## Этап 0. Подготовка: загрузка данных и проверка переноса моделей


### 0.1. Загрузка и валидация данных


In [1]:
import pandas as pd

from recsys import config, data, evaluation, split
from recsys.models import ContentBased, ItemCF, Popularity, PureSVD

pd.set_option("display.float_format", "{:.4f}".format)

dataset = data.load_all()
pd.Series(data.validate(dataset), name="value").to_frame()

,value
n_ratings,5976479
n_users,53424
n_books,10000
n_tags,34252
n_book_tags,999912
book_tags_duplicate_pairs,8
book_tags_nonpositive_count,6
books_missing_original_title,585


Проверка целостности данных завершается без критических нарушений: 8 дублирующихся пар (book_id, tag_name) и 6 записей с неположительным count в book_tags, 585 книг без original_title — некритические особенности, учитываемые при очистке тегов и построении текстовых профилей. Структура датасета (5 976 479 оценок, 53 424 пользователя, 10 000 книг) совпадает с ожидаемой по домашнему заданию.


### 0.2. Split train_inner / valid


In [2]:
outer = split.temporal_split(dataset.ratings)
inner = split.temporal_split(outer.train)
train_inner, valid = inner.train, inner.test
book_tags = data.clean_book_tags(dataset)
pd.Series({"train_inner": len(train_inner), "valid": len(valid)}, name="n_ratings")

train_inner    3824978
valid           956230
Name: n_ratings, dtype: int64

Train_inner и valid получены повторным применением per-user temporal split к обучающей части outer-разбиения, без обращения к отложенной test-выборке.


### 0.3. Smoke-проверка: nDCG@10 перенесённых моделей на valid


In [3]:
eval_set = evaluation.build_eval_set(train_inner, valid)

models = {
    "Popularity (weighted)": Popularity("weighted", config.POPULARITY_WR_M),
    "Content-Based": ContentBased(dataset.books, book_tags),
    "Item-based CF": ItemCF(),
    "PureSVD": PureSVD(config.PURE_SVD_N_FACTORS),
}
for model in models.values():
    model.fit(train_inner)

smoke_table = evaluation.compare(list(models.values()), eval_set, ks=(10,)).xs(10, level="k")
smoke_table[["ndcg"]]

,ndcg
model,
"Popularity[weighted, m=100000]",0.0441
Content-Based[tags=all],0.0536
"Item-CF[adjusted_cosine, K=50, λ=50, similarity]",0.0968
PureSVD[k=200],0.0733


nDCG@10 на valid для перенесённых моделей совпадает с результатами домашнего задания с точностью до 1e-4: Popularity (weighted) 0.0441, Content-Based 0.0536, Item-based CF 0.0968, PureSVD 0.0733. Совпадение чисел подтверждает корректность переноса кода, конфигурации протокола оценки и реализации split без изменений.
